# Alineamiento DPO Split-Backward (`Qwen/Qwen3-4B-Instruct-2507`) → Ollama `Q4_K_M`

A diferencia del SFT puro (que empujó el estilo hasta forzar `calcular` sobre fechas y voltear desigualdades), **DPO (*Direct Preference Optimization*)** compara explícitamente pares **`chosen` vs `rejected`** manteniendo un ancla KL (`beta=0.1`) contra la referencia base `Qwen/Qwen3-4B-Instruct-2507`.

### Diseño pensado para Colab T4 (15 GB) y Mac (8 GB RAM)
1. **Cero fuga de datos (`construir_dataset_dpo.py` + `antifuga.py`):** 62 pares (46 train / 16 val) aislados de las 61 unidades y 95 preguntas de evaluación.
2. **DPO Split-Backward exacto:** precomputa `ref_logps` en `no_grad` sobre el modelo base antes de acoplar LoRA (`0 MB` extra de modelo de referencia en VRAM) y desacopla los grafos de `chosen` y `rejected` conservando el gradiente matemático exacto + regularización NLL (`lambda_nll=0.05`).
3. **Selección del mejor checkpoint (`eval_dpo_loss`):** fusiona en FP16/BF16 el mejor adaptador medido en validación (`best_adapter`) antes de cuantizar a `GGUF Q4_K_M` para **`udec-agente:4b-dpo`**.

In [ ]:
#@title 1. GPU, repositorio (`exp/dpo-agente`) y dependencias
import os, subprocess
from pathlib import Path

!nvidia-smi

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
REPO_BRANCH = "exp/dpo-agente"
REPO_DIR = Path("/content/Proyecto_Git")

if not REPO_DIR.exists():
    res = subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], text=True)
    if res.returncode != 0:
        raise RuntimeError(f"Asegúrate de haber hecho push de la rama {REPO_BRANCH} a GitHub antes de ejecutar en Colab.")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)

os.chdir(REPO_DIR)
!pip uninstall -y -q torchao
!pip install -q "transformers>=4.51.0" "peft>=0.15.0" "bitsandbytes>=0.45.0" "accelerate>=1.0.0" gguf sentencepiece protobuf matplotlib pymupdf pdfplumber


In [ ]:
#@title 2. Regenerar dataset DPO antifuga, correr tests unitarios y validar tokenización
!python3 -m asistente_local.finetune.construir_dataset_dpo
!python3 -m unittest discover -s "asistente_local/tests" -v
!python3 -m asistente_local.finetune.entrenar_agente_dpo --validate-only


In [ ]:
#@title 3. Entrenar DPO Split-Backward (`beta=0.1`, `lambda_nll=0.05`, `lr=1e-5`, 3 épocas) y fusionar FP16/BF16
from IPython.display import Image, display
from pathlib import Path

OUT_DIR = Path("/content/dpo_salida")

!python3 -m asistente_local.finetune.entrenar_agente_dpo \
    --output-dir /content/dpo_salida \
    --merge-dir /content/dpo_merged_hf \
    --beta 0.1 \
    --lambda-nll 0.05 \
    --epochs 3.0 \
    --lr 1e-5 \
    --grad-accum 4 \
    --eval-every 6

if (OUT_DIR / "dpo_loss_curve.png").is_file():
    display(Image(filename=str(OUT_DIR / "dpo_loss_curve.png")))
print(Path(OUT_DIR / "training_metrics.json").read_text())


In [ ]:
#@title 4. Compilar `llama-quantize` (`-DLLAMA_CURL=OFF`) y exportar GGUF Q4_K_M + Modelfile Ollama
from pathlib import Path

LLAMA_CPP = Path("/content/llama.cpp")
if not LLAMA_CPP.exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -B /content/llama.cpp/build /content/llama.cpp -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize -j4

!python3 -m asistente_local.finetune.exportar_ollama \
    --merged-dir /content/dpo_merged_hf \
    --salida-dir /content/dpo_ollama_pkg \
    --llama-cpp-dir /content/llama.cpp \
    --nombre-base udec-agente-qwen3-4b-dpo \
    --cuantizacion Q4_K_M

!ls -lh /content/dpo_ollama_pkg


In [ ]:
#@title 5. Guardar en Google Drive (`UdeC_GenIA_DPO_Agente/`) para descargar al Mac
import shutil
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

dest = Path("/content/drive/MyDrive/UdeC_GenIA_DPO_Agente")
dest.mkdir(parents=True, exist_ok=True)
shutil.copytree("/content/dpo_ollama_pkg", dest / "ollama_pkg", dirs_exist_ok=True)
if Path("/content/dpo_salida/best_adapter").exists():
    shutil.copytree("/content/dpo_salida/best_adapter", dest / "best_adapter", dirs_exist_ok=True)
shutil.copytree("/content/dpo_salida/adapter", dest / "adapter", dirs_exist_ok=True)
shutil.copy("/content/dpo_salida/training_metrics.json", dest / "training_metrics.json")
shutil.copy("/content/dpo_salida/dpo_loss_curve.png", dest / "dpo_loss_curve.png")

# Empaquetar métricas ligeras por si se quieren bajar directo sin GGUF
ligero = Path("/content/dpo_metricas_pkg")
ligero.mkdir(parents=True, exist_ok=True)
shutil.copy("/content/dpo_salida/training_metrics.json", ligero / "training_metrics.json")
shutil.copy("/content/dpo_salida/dpo_loss_curve.png", ligero / "dpo_loss_curve.png")
shutil.copy("/content/dpo_ollama_pkg/Modelfile", ligero / "Modelfile")
shutil.make_archive("/content/dpo_metricas", "zip", root_dir=ligero)
shutil.copy("/content/dpo_metricas.zip", dest / "dpo_metricas.zip")
print("Guardado en Google Drive:", dest)

# En tu Mac, tras descargar `ollama_pkg/`:
#   cd ollama_pkg && ollama create udec-agente:4b-dpo -f Modelfile
#   python3 -m asistente_local.benchmark --dataset nuevas_20 --modelo udec-agente:4b-dpo
#   python3 -m asistente_local.benchmark --dataset dificil_25 --modelo udec-agente:4b-dpo
